In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('charts', exist_ok=True)

# --- Load both CSV files ---
print("Loading dataset...")
df_data   = pd.read_csv('data/data.csv', index_col=0)
df_labels = pd.read_csv('data/labels.csv', index_col=0)

print(f"✅ Data loaded!")
print(f"   Gene expression shape : {df_data.shape}")
print(f"   Labels shape          : {df_labels.shape}")
df_labels.columns = ['cancer_type']
df = df_data.copy()
df['cancer_type'] = df_labels['cancer_type'].values

print("Combined shape:", df.shape)
print("\nSample — last 3 columns:")
print(df.iloc[:3, -3:])

print("\nCancer types in dataset:")
print(df['cancer_type'].value_counts())

print("\nWhat each type means:")
cancer_info = {
    'BRCA': 'Breast Cancer',
    'KIRC': 'Kidney Clear Cell Carcinoma',
    'COAD': 'Colon Adenocarcinoma',
    'LUAD': 'Lung Adenocarcinoma',
    'PRAD': 'Prostate Cancer'
}
for code, name in cancer_info.items():
    count = (df['cancer_type'] == code).sum()
    print(f"  {code} = {name:35s} → {count} patients")
    print(f"Total missing values : {df.isnull().sum().sum()}")
print(f"Data type of genes   : {df.iloc[:,0].dtype}")
print(f"Duplicate rows       : {df.duplicated().sum()}")

print("\nStatistics on first 5 gene columns:")
print(df.iloc[:, :5].describe().round(3))

print("\n✅ Data quality check complete!")

In [ ]:
#LEAKAGE-FREE PREPROCESSING

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import json

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold

os.makedirs('charts', exist_ok=True)
os.makedirs('models', exist_ok=True)

df_data   = pd.read_csv('data/data.csv',   index_col=0)
df_labels = pd.read_csv('data/labels.csv', index_col=0)
df_labels.columns = ['cancer_type']

df = df_data.copy()
df['cancer_type'] = df_labels['cancer_type'].values

print(f"Raw data loaded: {df.shape}")
print(f"   Genes: {df.shape[1]-1}, Patients: {df.shape[0]}")

In [ ]:
total_missing = df.isnull().sum().sum()
print(f"Total missing values: {total_missing}")

if total_missing > 0:
    gene_cols = df.columns[:-1]
    df[gene_cols] = df[gene_cols].fillna(df[gene_cols].median())
    print("✅ Missing values filled")
else:
    print("✅ No missing values found")

In [ ]:


X_raw = df.drop(columns=['cancer_type'])
y_raw = df['cancer_type']

le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)

reverse_mapping = dict(zip(le.transform(le.classes_).tolist(),
                           le.classes_.tolist()))
class_names = [reverse_mapping[i] for i in range(5)]

print("Label encoding:")
for name, code in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {name} → {code}")

In [ ]:


X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"✅ Split done BEFORE any filtering/scaling:")
print(f"   X_train_raw : {X_train_raw.shape}")
print(f"   X_test_raw  : {X_test_raw.shape}")
print(f"\n   From this point on, X_test_raw is 'locked' —")
print(f"   we only ever .transform() it, never .fit() on it.")

In [ ]:


print("Applying variance threshold — fit on TRAIN only...")

selector = VarianceThreshold(threshold=0.1)

# FIT only on training data
selector.fit(X_train_raw)

# TRANSFORM both sets using the same fitted selector
X_train_filtered = selector.transform(X_train_raw)
X_test_filtered  = selector.transform(X_test_raw)

surviving_genes = X_train_raw.columns[selector.get_support()]

X_train_filtered = pd.DataFrame(X_train_filtered, columns=surviving_genes)
X_test_filtered  = pd.DataFrame(X_test_filtered,  columns=surviving_genes)

print(f" Variance filter fitted on train, applied to both:")
print(f"   Genes before: {X_train_raw.shape[1]}")
print(f"   Genes after : {X_train_filtered.shape[1]}")
print(f"   X_train_filtered: {X_train_filtered.shape}")
print(f"   X_test_filtered : {X_test_filtered.shape}")

In [ ]:

print("Applying StandardScaler — fit on TRAIN only...")

scaler = StandardScaler()
scaler.fit(X_train_filtered)

X_train_scaled = scaler.transform(X_train_filtered)
X_test_scaled  = scaler.transform(X_test_filtered)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=surviving_genes)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=surviving_genes)

print(f"Scaling fitted on train, applied to both:")
print(f"   Train mean (gene 1): {X_train_scaled.iloc[:,0].mean():.6f} (should be ~0)")
print(f"   Train std  (gene 1): {X_train_scaled.iloc[:,0].std():.6f}  (should be ~1)")
print(f"   Test mean  (gene 1): {X_test_scaled.iloc[:,0].mean():.6f}  (will NOT be exactly 0 — that's correct!)")
print(f"\n   Test set mean isn't exactly 0 because it's scaled using")
print(f"   TRAIN statistics, not its own — this is the honest,")
print(f"   leakage-free way to do it.")

In [ ]:
print("Saving leakage-free preprocessed data...")

np.save('models/X_train.npy', X_train_scaled.values)
np.save('models/X_test.npy',  X_test_scaled.values)
np.save('models/y_train.npy', y_train)
np.save('models/y_test.npy',  y_test)

joblib.dump(scaler,              'models/scaler.pkl')
joblib.dump(le,                  'models/label_encoder.pkl')
joblib.dump(selector,            'models/variance_selector.pkl')
joblib.dump(list(surviving_genes),'models/surviving_genes.pkl')

with open('models/label_mapping.json', 'w') as f:
    json.dump(reverse_mapping, f, indent=2)

print("✅ All corrected files saved to models/")
print(f"   X_train: {X_train_scaled.shape}")
print(f"   X_test : {X_test_scaled.shape}")
print(f"\n   This dataset is now leakage-free.")
print(f"   Same {X_train_scaled.shape[1]} genes as before (~19,276)")
print(f"   — variance filtering removes the same near-zero genes")
print(f"   whether fit on 801 or 640 patients, so the count barely changes.")
print(f"   What's different is HOW the values were scaled.")

In [ ]:
# ============================================================
# DAY 3.5 — 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import json
import os

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
import xgboost as xgb

os.makedirs('charts', exist_ok=True)

X_train = np.load('models/X_train.npy')
X_test  = np.load('models/X_test.npy')
y_train = np.load('models/y_train.npy')
y_test  = np.load('models/y_test.npy')

gene_names = joblib.load('models/surviving_genes.pkl')

with open('models/label_mapping.json', 'r') as f:
    label_mapping = json.load(f)
label_mapping = {int(k): v for k, v in label_mapping.items()}
class_names = [label_mapping[i] for i in range(5)]

print("✅ Loaded leakage-free preprocessed data")
print(f"   X_train: {X_train.shape}  (currently {X_train.shape[1]} genes)")

In [ ]:

print("Testing multiple gene counts with nested 5-fold CV...")
print("(This trains XGBoost ~30 times total — please wait)\n")

k_values = [100, 500, 1000, 2000, 5000, 10000, X_train.shape[1]]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

for k in k_values:
    k_actual = min(k, X_train.shape[1])  # can't select more than we have

    pipeline = Pipeline([
        ('feature_select', SelectKBest(f_classif, k=k_actual)),
        ('classifier', xgb.XGBClassifier(
            objective='multi:softprob',
            num_class=5,
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbosity=0,
            n_jobs=-1
        ))
    ])

    scores = cross_val_score(pipeline, X_train, y_train,
                              cv=cv, scoring='accuracy', n_jobs=1)

    results.append({
        'k_genes'      : k_actual,
        'mean_accuracy': scores.mean(),
        'std_accuracy' : scores.std()
    })

    print(f"  k={k_actual:>6} genes → "
          f"CV accuracy: {scores.mean()*100:.2f}% "
          f"(± {scores.std()*100:.2f}%)")

results_df = pd.DataFrame(results)
print("\n✅ Nested cross-validation complete across all gene counts")

In [ ]:


best_accuracy = results_df['mean_accuracy'].max()
threshold = best_accuracy - 0.005   # within 0.5%

candidates = results_df[results_df['mean_accuracy'] >= threshold]
chosen_k = int(candidates['k_genes'].min())

chosen_row = results_df[results_df['k_genes'] == chosen_k].iloc[0]

print("=" * 60)
print("  FEATURE COUNT SELECTION — DECISION")
print("=" * 60)
print(f"  Best CV accuracy found      : {best_accuracy*100:.2f}%")
print(f"  Threshold (best - 0.5%)     : {threshold*100:.2f}%")
print(f"  Smallest k meeting threshold: {chosen_k} genes")
print(f"  CV accuracy at chosen k     : {chosen_row['mean_accuracy']*100:.2f}% "
      f"(± {chosen_row['std_accuracy']*100:.2f}%)")
print("=" * 60)
print(f"\n  ★ DECISION: We will use {chosen_k} genes for the final model")
print(f"  ★ This reduces dimensionality by "
      f"{(1 - chosen_k/X_train.shape[1])*100:.1f}% "
      f"compared to the full {X_train.shape[1]}-gene set")
print(f"  ★ With only {(best_accuracy - chosen_row['mean_accuracy'])*100:.2f}% "
      f"accuracy difference from the best possible")

In [ ]:


final_selector = SelectKBest(f_classif, k=chosen_k)
final_selector.fit(X_train, y_train)

X_train_selected = final_selector.transform(X_train)
X_test_selected  = final_selector.transform(X_test)

# Get the actual gene names that were selected
selected_mask  = final_selector.get_support()
selected_genes = [gene_names[i] for i in range(len(gene_names)) if selected_mask[i]]

# Get F-scores for these genes (for reporting top genes)
f_scores = final_selector.scores_[selected_mask]

top_genes_df = pd.DataFrame({
    'gene'   : selected_genes,
    'f_score': f_scores
}).sort_values('f_score', ascending=False)

print(f"✅ Final feature selection applied:")
print(f"   X_train_selected: {X_train_selected.shape}")
print(f"   X_test_selected : {X_test_selected.shape}")
print(f"\nTop 15 genes by F-score (your strongest cancer-type predictors):")
print(top_genes_df.head(15).to_string(index=False))

In [ ]:

np.save('models/X_train_selected.npy', X_train_selected)
np.save('models/X_test_selected.npy',  X_test_selected)

joblib.dump(final_selector,  'models/anova_selector.pkl')
joblib.dump(selected_genes,  'models/selected_genes.pkl')

top_genes_df.to_csv('models/top_genes_by_fscore.csv', index=False)
results_df.to_csv('models/feature_selection_cv_results.csv', index=False)

print("✅ Saved:")
print(f"   models/X_train_selected.npy  ({X_train_selected.shape})")
print(f"   models/X_test_selected.npy   ({X_test_selected.shape})")
print(f"   models/anova_selector.pkl")
print(f"   models/selected_genes.pkl")
print(f"   models/top_genes_by_fscore.csv   ← paste into your paper")
print(f"   models/feature_selection_cv_results.csv ← elbow chart data")

In [ ]:
#reduced parametrs

import xgboost as xgb, numpy as np, pandas as pd, joblib, json, os
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

os.makedirs('models', exist_ok=True)
os.makedirs('charts', exist_ok=True)

X_train = np.load('models/X_train_selected.npy')
X_test  = np.load('models/X_test_selected.npy')
y_train = np.load('models/y_train.npy')
y_test  = np.load('models/y_test.npy')

with open('models/label_mapping.json') as f:
    label_mapping = {int(k): v for k, v in json.load(f).items()}
class_names = [label_mapping[i] for i in range(5)]

# ---- Test several simpler/more regularized configs ----
param_sets = {
    'original (deep)'   : dict(max_depth=6, n_estimators=300, learning_rate=0.1,
                                subsample=0.8, colsample_bytree=0.8,
                                reg_alpha=0, reg_lambda=1),
    'light_reg'          : dict(max_depth=4, n_estimators=200, learning_rate=0.1,
                                subsample=0.7, colsample_bytree=0.7,
                                reg_alpha=0.5, reg_lambda=2),
    'strong_reg'         : dict(max_depth=3, n_estimators=150, learning_rate=0.05,
                                subsample=0.6, colsample_bytree=0.6,
                                reg_alpha=1, reg_lambda=5),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, params in param_sets.items():
    model = xgb.XGBClassifier(objective='multi:softprob', num_class=5,
                               random_state=42, n_jobs=-1, eval_metric='mlogloss',
                               **params)
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    model.fit(X_train, y_train)
    train_acc = model.score(X_train, y_train)
    test_acc  = model.score(X_test, y_test)
    gap = train_acc - test_acc

    results.append({'config': name, 'cv_mean': cv_scores.mean(),
                     'train_acc': train_acc, 'test_acc': test_acc, 'gap': gap})
    print(f"{name:<15} CV={cv_scores.mean()*100:.2f}%  "
          f"Train={train_acc*100:.2f}%  Test={test_acc*100:.2f}%  "
          f"Gap={gap*100:.2f}%")

results_df = pd.DataFrame(results)

# ---- Pick the config with smallest train-test gap (least overfit) ----
best_name = results_df.loc[results_df['gap'].idxmin(), 'config']
best_params = param_sets[best_name]
print(f"\n★ Chosen config (smallest overfit gap): {best_name}")
print(best_params)

# ---- Train final model with chosen params ----
final_model = xgb.XGBClassifier(objective='multi:softprob', num_class=5,
                                 random_state=42, n_jobs=-1, eval_metric='mlogloss',
                                 **best_params)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nFinal Test Accuracy: {accuracy*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=class_names))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix — Regularized Model ({best_name})')
plt.tight_layout()
plt.savefig('charts/chart22_confusion_regularized.png', dpi=150)
plt.show()

# ---- Save ----
joblib.dump(final_model, 'models/xgboost_model.pkl')
final_model.save_model('models/xgboost_model.json')
results_df.to_csv('models/regularization_comparison.csv', index=False)

with open('models/training_results.json', 'w') as f:
    json.dump({'chosen_config': best_name, **best_params,
                'test_accuracy': float(accuracy)}, f, indent=2)

print("\n Saved regularized model → models/xgboost_model.pkl")
print("   This is the model SHAP/blockchain/dashboard will use")

In [ ]:

import subprocess
subprocess.run(['pip', 'install', 'shap', '--quiet'], capture_output=True)

import shap
print(f"SHAP version: {shap.__version__}")

major, minor = [int(x) for x in shap.__version__.split('.')[:2]]
if major == 0 and minor < 40:
    print("  SHAP version below 0.40 detected.")
    print("    Run: pip install shap --upgrade")
    print("    Some plot functions may behave differently.")
else:
    print(f" SHAP version {shap.__version__} — compatible")

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')
os.makedirs('shap_outputs', exist_ok=True)
model = joblib.load('models/xgboost_model.pkl')
print(f" Model loaded: {type(model).__name__}")
try:
    nc = model.n_classes_
    print(f"   Model classes: {nc}")
except:
    print("   (num_class readable from model config)")

X_test  = np.load('models/X_test_selected.npy')
y_test  = np.load('models/y_test.npy')
X_train = np.load('models/X_train_selected.npy')

selected_genes = joblib.load('models/selected_genes.pkl')

with open('models/label_mapping.json') as f:
    label_mapping = {int(k): v for k, v in json.load(f).items()}
class_names = [label_mapping[i] for i in range(5)]

X_test_df  = pd.DataFrame(X_test,  columns=selected_genes)
X_train_df = pd.DataFrame(X_train, columns=selected_genes)

print(f"\nData loaded:")
print(f"   X_test  shape: {X_test_df.shape}")
print(f"   X_train shape: {X_train_df.shape}")
print(f"   Genes        : {len(selected_genes)}")
print(f"   Classes      : {class_names}")

In [ ]:
print("Creating SHAP TreeExplainer...")
background = shap.sample(X_train_df, 100, random_state=42)
explainer = shap.TreeExplainer(
    model,
    data              = background,
    feature_perturbation = "interventional",
    model_output      = "raw"
)

print("Explainer created!")
print(f"\nBaseline raw scores per class (expected_value):")
print(f"(These come from training distribution — NOT 1/n_classes)")
for i, name in enumerate(class_names):
    print(f"  {name:<6}: {explainer.expected_value[i]:.4f}")


In [ ]:
print("Computing SHAP values for all test patients...")
raw_shap_output = explainer.shap_values(X_test_df)

print("Raw SHAP output inspection:")
print(f"  Type: {type(raw_shap_output)}")
if isinstance(raw_shap_output, list):
    print(f"  Format: LIST of {len(raw_shap_output)} arrays")
    print(f"  Each array shape: {raw_shap_output[0].shape}")
    print(f"  → Format A detected (one array per class)")
elif isinstance(raw_shap_output, np.ndarray):
    print(f"  Format: Single numpy array, shape: {raw_shap_output.shape}")
    print(f"  → Format B detected (3D array)")
if isinstance(raw_shap_output, list):
    # Format A — already the right structure
    shap_values_list = raw_shap_output
    print("\nUsing Format A directly")

elif isinstance(raw_shap_output, np.ndarray):
    if raw_shap_output.ndim == 3:
        shap_values_list = [
            raw_shap_output[:, :, i]
            for i in range(raw_shap_output.shape[2])
        ]
        print("\n Converted Format B → standard list format")
    else:
        raise ValueError(f"Unexpected SHAP array shape: {raw_shap_output.shape}")
else:
    raise TypeError(f"Unexpected SHAP output type: {type(raw_shap_output)}")

n_classes  = len(shap_values_list)
n_patients = shap_values_list[0].shape[0]
n_genes    = shap_values_list[0].shape[1]

print(f"\nFinal normalised SHAP structure:")
print(f"  Classes   : {n_classes}  (one array per cancer type)")
print(f"  Patients  : {n_patients} (test set size)")
print(f"  Genes     : {n_genes} (selected gene features)")

assert n_classes  == 5,                      "Expected 5 cancer classes"
assert n_patients == X_test_df.shape[0],     "Patient count mismatch"
assert n_genes    == X_test_df.shape[1],     "Gene count mismatch"
print("\n All dimension checks passed")

np.save('shap_outputs/shap_values.npy', np.array(shap_values_list))
joblib.dump(explainer, 'shap_outputs/explainer.pkl')
print(" Saved → shap_outputs/shap_values.npy")
print(" Saved → shap_outputs/explainer.pkl")

In [ ]:
# ============================================================
# WHY THIS STEP:
# SHAP calculations can silently produce NaN values if the
# model has issues or the background data was problematic.
# We check explicitly before plotting — a NaN in SHAP values
# will cause charts to appear blank or crash.
# ============================================================

print("Running SHAP value integrity checks...")

all_passed = True
for i, cancer in enumerate(class_names):
    sv = shap_values_list[i]

    has_nan  = np.isnan(sv).any()
    has_inf  = np.isinf(sv).any()
    correct_shape = (sv.shape == (n_patients, n_genes))

    status = "✅" if (not has_nan and not has_inf and correct_shape) else "❌"
    if not (not has_nan and not has_inf and correct_shape):
        all_passed = False

    print(f"  {cancer}: shape={sv.shape}  "
          f"NaN={has_nan}  Inf={has_inf}  {status}")

if all_passed:
    print("\n✅ All SHAP values are valid — ready for plotting")
else:
    print("\n❌ Issues detected — do NOT proceed to plotting")
    print("   Troubleshooting:")
    print("   1. Check your model loaded correctly")
    print("   2. Verify X_test_selected.npy matches the model's expected input")
    print("   3. Try: explainer = shap.TreeExplainer(model) without background data")
    print("   4. Check SHAP version: pip install shap --upgrade")

In [ ]:

print("Generating per-class beeswarm charts...")
colors = ['#534AB7','#1D9E75','#D85A30','#BA7517','#D4537E']

for i, (cancer, color) in enumerate(zip(class_names, colors)):
    print(f"  Plotting {cancer}...")
    fig, ax = plt.subplots(figsize=(10, 6))

    shap.summary_plot(
        shap_values_list[i],
        X_test_df,
        plot_type   = "dot",        # beeswarm style
        max_display = 15,
        show        = False,
        color_bar   = True
    )

    plt.title(
        f'SHAP Beeswarm — {cancer}\n'
        f'Top 15 genes | x-axis = SHAP value (log-odds space)\n'
        f'Red=high expression, Blue=low expression',
        fontsize=11, fontweight='500', pad=10
    )
    plt.tight_layout()
    fname = f'shap_outputs/shap_beeswarm_{cancer}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   Saved → {fname}")

print("\n All 5 beeswarm charts saved")

In [ ]:


fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (cancer, color) in enumerate(zip(class_names, colors)):
    mean_abs_shap = np.abs(shap_values_list[i]).mean(axis=0)
    top_idx = np.argsort(mean_abs_shap)[::-1][:12]

    axes[i].barh(
        range(12),
        mean_abs_shap[top_idx][::-1],
        color=color, edgecolor='white', linewidth=0.5
    )
    axes[i].set_yticks(range(12))
    axes[i].set_yticklabels(
        [selected_genes[j][:14] for j in top_idx[::-1]],
        fontsize=8
    )
    axes[i].set_title(f'{cancer}', fontweight='500',
                      color=color, fontsize=12)
    axes[i].set_xlabel('Mean |SHAP value| (log-odds)', fontsize=9)
    axes[i].axvline(x=0, color='gray', linewidth=0.5)

# Hide the unused 6th subplot
axes[5].set_visible(False)

plt.suptitle(
    'Top 12 Most Important Genes per Cancer Type\n'
    '(Mean absolute SHAP value in log-odds space)',
    fontsize=14, fontweight='500', y=1.02
)
plt.tight_layout()
plt.savefig('shap_outputs/shap_bar_per_class.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved → shap_outputs/shap_bar_per_class.png")

In [ ]:

patient_idx  = 0   # change this to explain any patient
pred_class   = int(model.predict(X_test_df.iloc[[patient_idx]])[0])
actual_class = int(y_test[patient_idx])
pred_proba   = model.predict_proba(X_test_df.iloc[[patient_idx]])[0]

print(f"Explaining Patient #{patient_idx}:")
print(f"  Actual cancer type    : {class_names[actual_class]}")
print(f"  Predicted cancer type : {class_names[pred_class]}")
print(f"  Model confidence      : {pred_proba[pred_class]*100:.1f}%  "
      f"(from predict_proba — probability space)")
print(f"  Raw model score       : "
      f"{np.sum(shap_values_list[pred_class][patient_idx]) + explainer.expected_value[pred_class]:.4f}  "
      f"(log-odds space — what SHAP explains)")
print(f"  Correct?              : "
      f"{' Yes' if pred_class == actual_class else '❌ No'}\n")

# Build the Explanation object for the predicted class
shap_vals_patient = shap_values_list[pred_class][patient_idx]
baseline_val      = explainer.expected_value[pred_class]

# Verify SHAP values sum correctly
computed_output = baseline_val + np.sum(shap_vals_patient)
print(f"  SHAP integrity check:")
print(f"    baseline + sum(SHAP) = {computed_output:.4f}")
print(f"    (This should equal the model's raw score above)\n")

explanation = shap.Explanation(
    values        = shap_vals_patient,
    base_values   = baseline_val,
    data          = X_test_df.iloc[patient_idx].values,
    feature_names = list(selected_genes)
)

plt.figure(figsize=(10, 7))
shap.plots.waterfall(explanation, max_display=12, show=False)
plt.title(
    f'Patient #{patient_idx} — Predicted: {class_names[pred_class]} '
    f'({pred_proba[pred_class]*100:.1f}% confidence)\n'
    f'x-axis values are in log-odds (raw score) space — not probabilities',
    fontsize=10, fontweight='500'
)
plt.tight_layout()
plt.savefig('shap_outputs/shap_waterfall_patient0.png',
            dpi=150, bbox_inches='tight')
plt.show()
print(" Saved → shap_outputs/shap_waterfall_patient0.png")

In [ ]:

print("Extracting top genes per cancer type...\n")

top_genes_per_class = {}

for i, cancer in enumerate(class_names):
    sv          = shap_values_list[i]
    mean_abs    = np.abs(sv).mean(axis=0)
    mean_signed = sv.mean(axis=0)
    top_idx     = np.argsort(mean_abs)[::-1][:5]

    top_genes_per_class[cancer] = [
        {
            'rank'              : rank + 1,
            'gene'              : selected_genes[j],
            'mean_abs_shap'     : round(float(mean_abs[j]), 5),
            'mean_signed_shap'  : round(float(mean_signed[j]), 5),
            'direction'         : (
                'high expression → increases score'
                if mean_signed[j] > 0
                else 'high expression → decreases score'
            ),
            'note'              : 'log-odds space, not probability'
        }
        for rank, j in enumerate(top_idx)
    ]

# Pretty print
for cancer, genes in top_genes_per_class.items():
    print(f"  {cancer}:")
    for g in genes:
        print(f"    {g['rank']}. {g['gene']:<22} "
              f"mean|SHAP|={g['mean_abs_shap']:.4f}  "
              f"({g['direction'][:30]})")
    print()

# Save
with open('shap_outputs/top_genes_per_class.json', 'w') as f:
    json.dump(top_genes_per_class, f, indent=2)

rows = []
for cancer, genes in top_genes_per_class.items():
    for g in genes:
        rows.append({'cancer_type': cancer, **g})
pd.DataFrame(rows).to_csv(
    'shap_outputs/top_genes_per_class.csv', index=False)

print(" Saved → shap_outputs/top_genes_per_class.json")
print(" Saved → shap_outputs/top_genes_per_class.csv")
print("   Use these as 'model-identified predictive features' in your paper")

In [ ]:

def explain_patient(patient_idx, X_df=None, save=True):

    if X_df is None:
        X_df = X_test_df

    pred_class = int(model.predict(X_df.iloc[[patient_idx]])[0])
    pred_proba = model.predict_proba(X_df.iloc[[patient_idx]])[0]
    confidence = float(pred_proba[pred_class])

    if confidence >= 0.85:
        confidence_level = 'High Confidence'
    elif confidence >= 0.60:
        confidence_level = 'Moderate Confidence'
    else:
        confidence_level = 'Low Confidence — Needs Review'

    sv      = shap_values_list[pred_class][patient_idx]
    top_idx = np.argsort(np.abs(sv))[::-1][:5]

    result = {
        'patient_idx'      : int(patient_idx),
        'predicted_cancer' : class_names[pred_class],
        'confidence_pct'   : round(confidence * 100, 2),
        'confidence_level': confidence_level,
        'shap_note'        : (
            'SHAP values are in log-odds (raw score) space. '
            'Positive = increased raw score for predicted class. '
            'Not directly interpretable as probability contributions.'
        ),
        'top_5_genes': [
            {
                'gene'       : selected_genes[j],
                'shap_value' : round(float(sv[j]), 5),
                'direction'  : (
                    'toward prediction'
                    if sv[j] > 0
                    else 'away from prediction'
                ),
                'magnitude'  : 'strong' if abs(sv[j]) > 0.5 else 'moderate'
            }
            for j in top_idx
        ]
    }

    if save:
        path = f'shap_outputs/explanation_patient{patient_idx}.json'
        with open(path, 'w') as f:
            json.dump(result, f, indent=2)

    return result


print("Testing explain_patient() on first 3 patients:\n")
for idx in range(3):
    r = explain_patient(idx)
    print(f"  Patient {idx}: {r['predicted_cancer']} "
      f"({r['confidence_pct']}% confidence, {r['confidence_level']})")
    print(f"    Top gene: {r['top_5_genes'][0]['gene']} "
          f"SHAP={r['top_5_genes'][0]['shap_value']} "
          f"({r['top_5_genes'][0]['direction']}, "
          f"{r['top_5_genes'][0]['magnitude']})\n")

print(" explain_patient() function working")


In [ ]:
patient_idx = 0
class_idx = 2   # KIRC

baseline = explainer.expected_value[class_idx]

shap_sum = shap_values_list[class_idx][patient_idx].sum()

print("Baseline:", baseline)
print("Total SHAP:", shap_sum)
print("Baseline + SHAP:", baseline + shap_sum)

In [ ]:
0x53c36982d7ef3f17b556B3eF5Ac0B0dE6ee00de2

In [ ]:
#npx hardhat run scripts/deploy.js --network ganache

In [ ]:
from web3 import Web3
import json
import hashlib
import numpy as np
import pandas as pd
import joblib
import os
import time
import warnings
warnings.filterwarnings('ignore')

os.makedirs('blockchain', exist_ok=True)

# ---- Load contract info saved by deploy.js ----
with open('blockchain/contract_info.json', 'r') as f:
    contract_info = json.load(f)

CONTRACT_ADDRESS = contract_info['contractAddress']
CONTRACT_ABI     = contract_info['abi']
DEPLOYER_WALLET  = contract_info['deployerWallet']

print("Contract info loaded:")
print(f"  Address        : {CONTRACT_ADDRESS}")
print(f"  Deployer wallet: {DEPLOYER_WALLET}")
print(f"  Network        : {contract_info['network']}")

# ---- Connect to Ganache ----
w3 = Web3(Web3.HTTPProvider("http://127.0.0.1:7545"))

if w3.is_connected():
    print(f"\n✅ Connected to Ganache!")
    print(f"   Chain ID    : {w3.eth.chain_id}")
    print(f"   Block number: {w3.eth.block_number}")
    balance = w3.from_wei(
        w3.eth.get_balance(DEPLOYER_WALLET), 'ether'
    )
    print(f"   Wallet ETH  : {balance}")
else:
    print("❌ Not connected — make sure Ganache is running")
    raise ConnectionError("Cannot connect to Ganache")

In [ ]:
contract = w3.eth.contract(
    address=Web3.to_checksum_address(CONTRACT_ADDRESS),
    abi=CONTRACT_ABI
)

print(" Contract loaded!")
print(f"   Contract address: {contract.address}")

try:
    owner = contract.functions.owner().call()
    count = contract.functions.predictionCount().call()
    print(f"   Contract owner  : {owner}")
    print(f"   Records stored  : {count}")
    print(f"\n✅ Smart contract is live and responding")
except Exception as e:
    print(f" Contract not responding: {e}")
    print("   Redeploy: npx hardhat run scripts/deploy.js --network ganache")

In [ ]:
model          = joblib.load('models/xgboost_model.pkl')
selected_genes = joblib.load('models/selected_genes.pkl')
X_test         = np.load('models/X_test_selected.npy')
y_test         = np.load('models/y_test.npy')
shap_values    = np.load(
    'shap_outputs/shap_values.npy',
    allow_pickle=True
)

with open('models/label_mapping.json') as f:
    label_mapping = {int(k): v for k, v in json.load(f).items()}
class_names = [label_mapping[i] for i in range(5)]

X_test_df = pd.DataFrame(X_test, columns=selected_genes)

print(" AI pipeline loaded:")
print(f"   Model          : XGBoost")
print(f"   Test patients  : {X_test_df.shape[0]}")
print(f"   Genes          : {X_test_df.shape[1]}")
print(f"   Classes        : {class_names}")

In [ ]:
def hash_gene_vector(gene_vector):
    """
    SHA256 hash of patient's 100 gene values.
    Converts numpy array → JSON string → hash bytes.
    Returns bytes32 compatible with Solidity.
    """
    gene_list  = [round(float(v), 6) for v in gene_vector]
    gene_str   = json.dumps(gene_list, separators=(',', ':'))
    hash_bytes = hashlib.sha256(gene_str.encode()).digest()
    return hash_bytes  


def hash_shap_explanation(explanation_dict):
    """
    SHA256 hash of the SHAP explanation dictionary.
    Returns bytes32 compatible with Solidity.
    """
    shap_str   = json.dumps(explanation_dict,
                             sort_keys=True,
                             separators=(',', ':'))
    hash_bytes = hashlib.sha256(shap_str.encode()).digest()
    return hash_bytes


test_vec  = X_test_df.iloc[0].values
test_hash = hash_gene_vector(test_vec)
print(f" Hash functions working")
print(f"   Gene hash (hex): {test_hash.hex()}")
print(f"   Length         : {len(test_hash)} bytes (correct for bytes32)")

In [ ]:
def explain_patient(patient_idx):
    """
    Generate SHAP explanation for one patient.
    Returns dict with prediction + top genes.
    """
    # Handle both list and 3D array SHAP formats
    raw = shap_values
    if isinstance(raw, np.ndarray) and raw.ndim == 4:
        shap_list = [raw[i] for i in range(raw.shape[0])]
    elif isinstance(raw, np.ndarray) and raw.ndim == 3:
        shap_list = [raw[i] for i in range(raw.shape[0])]
    else:
        shap_list = list(raw)

    pred_class = int(model.predict(
        X_test_df.iloc[[patient_idx]])[0])
    pred_proba = model.predict_proba(
        X_test_df.iloc[[patient_idx]])[0]
    confidence = float(pred_proba[pred_class])

    if confidence >= 0.85:
        risk = 'High Risk'
    elif confidence >= 0.60:
        risk = 'Moderate Risk'
    else:
        risk = 'Low Confidence'

    sv      = shap_list[pred_class][patient_idx]
    top_idx = np.argsort(np.abs(sv))[::-1][:5]

    return {
        'patient_idx'      : int(patient_idx),
        'predicted_cancer' : class_names[pred_class],
        'confidence_pct'   : round(confidence * 100, 2),
        'risk_level'       : risk,
        'top_5_genes'      : [
            {
                'gene'       : selected_genes[j],
                'shap_value' : round(float(sv[j]), 5),
                'direction'  : (
                    'toward' if sv[j] > 0 else 'away from'
                )
            }
            for j in top_idx
        ]
    }

# Test
r = explain_patient(0)
print(f"✅ explain_patient() working")
print(f"   Patient 0: {r['predicted_cancer']} "
      f"({r['confidence_pct']}%, {r['risk_level']})")

In [ ]:
def store_prediction_on_chain(patient_idx):
    """
    Full pipeline:
    gene data → XGBoost → SHAP → hash → blockchain

    Returns transaction receipt with block number and tx hash.
    """
    print(f"\nProcessing Patient #{patient_idx}...")

    gene_vector = X_test_df.iloc[patient_idx].values

    explanation = explain_patient(patient_idx)

    predicted_cancer = explanation['predicted_cancer']
    confidence_int   = int(explanation['confidence_pct'])
    risk_level       = explanation['risk_level']
    top_genes        = explanation['top_5_genes']

    print(f"  Prediction  : {predicted_cancer} "
          f"({confidence_int}%, {risk_level})")

    gene_hash = hash_gene_vector(gene_vector)
    shap_hash = hash_shap_explanation(explanation)

    print(f"  Gene hash   : {gene_hash.hex()[:20]}...")
    print(f"  SHAP hash   : {shap_hash.hex()[:20]}...")

    offchain_dir = 'blockchain/offchain'
    os.makedirs(offchain_dir, exist_ok=True)

    gene_record = {
        'patient_idx' : patient_idx,
        'gene_values' : {
            selected_genes[i]: round(float(gene_vector[i]), 6)
            for i in range(len(selected_genes))
        },
        'hash'        : gene_hash.hex()
    }
    with open(f'{offchain_dir}/patient_{patient_idx}_genes.json','w') as f:
        json.dump(gene_record, f, indent=2)

    # Save SHAP explanation off-chain
    with open(f'{offchain_dir}/patient_{patient_idx}_shap.json','w') as f:
        json.dump(explanation, f, indent=2)

    # ---- Step E: Build transaction and send to blockchain ----
    print(f"  Sending transaction to Ganache...")

    # Get current nonce (transaction count for this wallet)
    nonce = w3.eth.get_transaction_count(DEPLOYER_WALLET)

    # Build the transaction
    tx = contract.functions.storePrediction(
        patient_idx,                    # uint patientId
        predicted_cancer,               # string predictedCancer
        confidence_int,                 # uint confidencePct
        risk_level,                     # string riskLevel
        top_genes[0]['gene'],           # string topGene1
        top_genes[1]['gene'],           # string topGene2
        top_genes[2]['gene'],           # string topGene3
        gene_hash,                      # bytes32 geneInputHash
        shap_hash                       # bytes32 shapExplanationHash
    ).build_transaction({
        'from'    : DEPLOYER_WALLET,
        'nonce'   : nonce,
        'gas'     : 500000,
        'gasPrice': w3.to_wei('20', 'gwei')
    })

    # Sign transaction with private key from .env
    private_key = os.getenv('GANACHE_PRIVATE_KEY')
    if not private_key:
        # Try loading .env manually
        env_path = 'blockchain_contracts/.env'
        if os.path.exists(env_path):
            with open(env_path) as f:
                for line in f:
                    if 'GANACHE_PRIVATE_KEY' in line:
                        private_key = line.split('=')[1].strip()
                        break

    signed_tx = w3.eth.account.sign_transaction(tx, private_key)

    # Send to Ganache
    tx_hash   = w3.eth.send_raw_transaction(
        signed_tx.raw_transaction)
    receipt   = w3.eth.wait_for_transaction_receipt(tx_hash)

    print(f"   Transaction mined!")
    print(f"     TX hash    : {receipt.transactionHash.hex()}")
    print(f"     Block #    : {receipt.blockNumber}")
    print(f"     Gas used   : {receipt.gasUsed}")
    print(f"     Status     : "
          f"{'SUCCESS' if receipt.status == 1 else 'FAILED'}")

    return {
        'patient_idx'     : patient_idx,
        'predicted_cancer': predicted_cancer,
        'confidence_pct'  : confidence_int,
        'tx_hash'         : receipt.transactionHash.hex(),
        'block_number'    : receipt.blockNumber,
        'gas_used'        : receipt.gasUsed,
        'status'          : receipt.status
    }

print(" store_prediction_on_chain() defined")

In [ ]:
# ---- Test with patient 0 first ----
print("Storing Patient 0 as test...")
result = store_prediction_on_chain(0)

print(f"\n{'='*50}")
print(f"  PATIENT 0 STORED ON BLOCKCHAIN")
print(f"{'='*50}")
print(f"  Cancer type  : {result['predicted_cancer']}")
print(f"  Confidence   : {result['confidence_pct']}%")
print(f"  TX hash      : {result['tx_hash'][:25]}...")
print(f"  Block number : {result['block_number']}")
print(f"  Gas used     : {result['gas_used']}")
print(f"{'='*50}")



In [ ]:
def get_record_from_chain(patient_idx):
    """
    Reads a stored prediction directly from the blockchain.
    This is a READ operation — costs no gas.
    """
    exists = contract.functions.recordExists(patient_idx).call()
    if not exists:
        print(f"No record found for patient {patient_idx}")
        return None

    result = contract.functions.getPrediction(patient_idx).call()

    record = {
        'patient_id'    : result[0],
        'cancer_type'   : result[1],
        'confidence'    : result[2],
        'risk_level'    : result[3],
        'top_gene_1'    : result[4],
        'top_gene_2'    : result[5],
        'top_gene_3'    : result[6],
        'gene_hash'     : result[7].hex(),
        'shap_hash'     : result[8].hex(),
        'timestamp'     : result[9],
        'submitted_by'  : result[10]
    }
    return record


# Read back patient 0
print("Reading Patient 0 record from blockchain...\n")
record = get_record_from_chain(0)

if record:
    print(f"{'='*55}")
    print(f"  RECORD FROM BLOCKCHAIN")
    print(f"{'='*55}")
    print(f"  Patient ID   : {record['patient_id']}")
    print(f"  Cancer type  : {record['cancer_type']}")
    print(f"  Confidence   : {record['confidence']}%")
    print(f"  Risk level   : {record['risk_level']}")
    print(f"  Top gene 1   : {record['top_gene_1']}")
    print(f"  Top gene 2   : {record['top_gene_2']}")
    print(f"  Top gene 3   : {record['top_gene_3']}")
    print(f"  Gene hash    : {record['gene_hash'][:25]}...")
    print(f"  SHAP hash    : {record['shap_hash'][:25]}...")
    print(f"  Submitted by : {record['submitted_by']}")
    print(f"{'='*55}")
    print(f"\n✅ Data successfully retrieved from blockchain!")

In [ ]:
def store_batch(patient_indices):
    """Store multiple patients on blockchain."""
    print(f"Storing {len(patient_indices)} patients on blockchain...\n")
    results = []
    for idx in patient_indices:
        try:
            r = store_prediction_on_chain(idx)
            results.append(r)
            time.sleep(0.5)   # small delay between transactions
        except Exception as e:
            print(f"  ❌ Patient {idx} failed: {e}")
    return results

# Store patients 1 to 4
batch_results = store_batch([1, 2, 3, 4])

print(f"\n{'='*55}")
print(f"  BATCH STORAGE COMPLETE")
print(f"{'='*55}")
for r in batch_results:
    print(f"  Patient {r['patient_idx']:>3} → "
          f"{r['predicted_cancer']:<6} "
          f"({r['confidence_pct']}%) "
          f"Block #{r['block_number']}")
print(f"{'='*55}")

# Verify count on chain
count = contract.functions.predictionCount().call()
print(f"\n  Total records on chain: {count}")

In [ ]:
print("TAMPER DETECTION DEMONSTRATION")
print("="*55)

# ---- Load the original gene data for patient 0 ----
with open('blockchain/offchain/patient_0_genes.json') as f:
    original_genes = json.load(f)

original_hash = hash_gene_vector(
    X_test_df.iloc[0].values)

# ---- Verify original data — should PASS ----
valid, msg = contract.functions.verifyGeneInput(
    0, original_hash
).call()
print(f"\nStep 1 — Verify ORIGINAL gene data:")
print(f"  Result  : {'✅ ' + msg if valid else '❌ ' + msg}")

# ---- Simulate tampering by modifying one gene value ----
print(f"\nStep 2 — Simulating tampering...")
print(f"  Changing first gene value slightly...")
tampered_vector = X_test_df.iloc[0].values.copy()
tampered_vector[0] = tampered_vector[0] + 0.001  # tiny change
tampered_hash = hash_gene_vector(tampered_vector)
print(f"  Original hash : {original_hash.hex()[:25]}...")
print(f"  Tampered hash : {tampered_hash.hex()[:25]}...")
print(f"  Hashes match  : {original_hash == tampered_hash}")

# ---- Verify tampered data — should FAIL ----
valid, msg = contract.functions.verifyGeneInput(
    0, tampered_hash
).call()
print(f"\nStep 3 — Verify TAMPERED gene data:")
print(f"  Result  : {'✅ ' + msg if valid else '❌ ' + msg}")

print(f"\n{'='*55}")
print(f"  DEMONSTRATION COMPLETE")
print(f"  Even a 0.001 change in one gene value")
print(f"  is immediately detected by the blockchain")
print(f"  This proves tamper-evident storage is working")
print(f"{'='*55}")

In [ ]:
# ---- Save complete audit log as JSON ----
print("Generating complete blockchain audit log...")

all_ids = contract.functions.getAllPatientIds().call()
audit_log = []

for pid in all_ids:
    rec = get_record_from_chain(pid)
    if rec:
        audit_log.append(rec)

with open('blockchain/audit_log.json', 'w') as f:
    json.dump(audit_log, f, indent=2)

print(f"✅ Audit log saved → blockchain/audit_log.json")
print(f"   Records in log: {len(audit_log)}")
print(f"\nSummary:")
print(f"  {'Patient':>8} {'Cancer':<8} "
      f"{'Confidence':>10} {'Block':>7}")
print(f"  {'-'*40}")
for r in audit_log:
    print(f"  {r['patient_id']:>8} {r['cancer_type']:<8} "
          f"{r['confidence']:>9}% {r['gene_hash'][:8]:>7}...")

In [ ]:
# PASTE IN NOTEBOOK — blockchain debug cell
from web3 import Web3
import json, os

# 1. Check connection
w3 = Web3(Web3.HTTPProvider("http://127.0.0.1:7545"))
print(f"Connected: {w3.is_connected()}")

# 2. Load contract
with open('blockchain/contract_info.json') as f:
    info = json.load(f)
print(f"Contract address: {info['contractAddress']}")

contract = w3.eth.contract(
    address=Web3.to_checksum_address(info['contractAddress']),
    abi=info['abi']
)

# 3. Check owner
owner = contract.functions.owner().call()
print(f"Contract owner: {owner}")

# 4. Check your private key → wallet address
with open('blockchain_contracts/.env') as f:
    for line in f:
        if 'GANACHE_PRIVATE_KEY' in line:
            key = line.split('=',1)[1].strip()

account = w3.eth.account.from_key(key)
print(f"Your wallet   : {account.address}")
print(f"Keys match    : {owner.lower() == account.address.lower()}")

# 5. Check balance
bal = w3.from_wei(
    w3.eth.get_balance(account.address), 'ether')
print(f"Balance       : {bal} ETH")

NameError: name 'model' is not defined